In [12]:
import pandas as pd 

df_vendas = pd.read_excel('Prova+Excel.xlsx',sheet_name='Base Vendas')
df_produto = pd.read_excel('Prova+Excel.xlsx',sheet_name='Cadastro Produto')
df_clientes = pd.read_excel('Prova+Excel.xlsx',sheet_name='Cadastro Clientes')

print(df_vendas.columns)
print(df_produto.columns)
print(df_clientes.columns)



Index(['SKU', 'Qtd Vendida', 'Loja', 'Data da Venda', 'Código Cliente'], dtype='str')
Index(['SKU', 'Produto', 'Marca', 'Tipo do Produto', 'Preco Unitario',
       'Custo Unitario'],
      dtype='str')
Index(['Código Cliente', 'Nome Cliente', 'Sexo', 'Nº de Filhos'], dtype='str')


In [10]:
df_produto

,SKU,Produto,Marca,Tipo do Produto,Preco Unitario,Custo Unitario
0,HL1001,Smart TV 50' 4K - Preto,LG,Televisão,2600.0,1700.0
1,HL1002,iPhone 7 - Dourado,Apple,Celular,2500.0,1500.0
2,HL1003,Galaxy S10 - Cinza,Samsung,Celular,4500.0,2800.0
3,HL1004,Apple Watch - Preto,Apple,Smart Watch,1750.0,900.0
4,HL1005,Câmera Digital Rebel T6 - Preto,Canon,Câmera,1500.0,850.0
5,HL1006,TV LED 32' - Preto,Samsung,Televisão,1400.0,900.0
6,HL1007,Inspiron 15 - Prata,Dell,Notebook,2300.0,1200.0
7,HL1008,Smart TV LED Full HD 55' - Preto,Philco,Televisão,2000.0,1250.0
8,HL1009,Moto G7 - Vermelho,Motorola,Celular,1400.0,750.0
9,HL1010,iPhone 6S - Rosa,Apple,Celular,1900.0,1150.0


In [7]:
# Agrupando por SKU e somando a coluna Qtd Vendida
df_sku_agrupado = df_vendas.groupby('SKU')['Qtd Vendida'].sum().reset_index()

# Exibindo o resultado
df_sku_agrupado


,SKU,Qtd Vendida
0,HL1001,3030
1,HL1002,3081
2,HL1003,3001
3,HL1004,2864
4,HL1005,2955
5,HL1006,3024
6,HL1007,3048
7,HL1008,2958
8,HL1009,2852
9,HL1010,2903


In [9]:
# 1. Unir as tabelas (Vendas + Cadastro) usando o SKU como chave
df_consolidado = df_vendas.merge(df_produto, on='SKU')

# 2. Calcular o Faturamento e o Custo Total de cada linha
df_consolidado['Faturamento'] = df_consolidado['Qtd Vendida'] * df_consolidado['Preco Unitario']
df_consolidado['Custo Total'] = df_consolidado['Qtd Vendida'] * df_consolidado['Custo Unitario']

# 3. Calcular o Lucro Total (Faturamento - Custo)
lucro_total = df_consolidado['Faturamento'].sum() - df_consolidado['Custo Total'].sum()

# Exibir o resultado formatado
print(f"O lucro total da operação é: R$ {lucro_total:,.2f}")

O lucro total da operação é: R$ 76,374,300.00


In [11]:
# 1. Unir a base de vendas com a base de produtos para ter os preços e as marcas
df_consolidado = df_vendas.merge(df_produto, on='SKU')

# 2. Calcular o faturamento de cada venda (Quantidade * Preço Unitário)
df_consolidado['Faturamento'] = df_consolidado['Qtd Vendida'] * df_consolidado['Preco Unitario']

# 3. Filtrar apenas os produtos da marca Apple
df_apple = df_consolidado[df_consolidado['Marca'] == 'Apple']

# 4. Agrupar por Produto, somar o faturamento e ordenar do maior para o menor
ranking_apple = df_apple.groupby('Produto')['Faturamento'].sum().sort_values(ascending=False).reset_index()

# 5. Exibir os dois primeiros
print(ranking_apple.head(2))

              Produto  Faturamento
0   iPhone XS - Preto   19896500.0
1  iPhone 7 - Dourado    7702500.0


In [14]:
# 1. Unir as 3 tabelas
df_consolidado = df_vendas.merge(df_produto[['SKU', 'Preco Unitario']], on='SKU')
df_consolidado = df_consolidado.merge(df_clientes[['Código Cliente', 'Sexo', 'Nº de Filhos']], on='Código Cliente')

# 2. Criar a lógica de faturamento com desconto
# Regra: Mulher (Feminino) E com pelo menos 1 filho (Nº de Filhos > 0)
def aplicar_desconto(linha):
    preco = linha['Preco Unitario']
    qtd = linha['Qtd Vendida']
    if linha['Sexo'] == 'Feminino' and linha['Nº de Filhos'] > 0:
        return (preco * 0.85) * qtd # 15% de desconto
    else:
        return preco * qtd

# Aplicando a função
df_consolidado['Faturamento_Final'] = df_consolidado.apply(aplicar_desconto, axis=1)

# 3. Agrupar e calcular as porcentagens
faturamento_sexo = df_consolidado.groupby('Sexo')['Faturamento_Final'].sum()
total_geral = faturamento_sexo.sum()

pct_masc = (faturamento_sexo['M'] / total_geral) * 100
pct_fem = (faturamento_sexo['F'] / total_geral) * 100

print(f"Resultado: Masculino {pct_masc:.1f}% ; Feminino {pct_fem:.1f}%")

Resultado: Masculino 54.6% ; Feminino 45.4%


In [15]:
# 1. Garantir que a coluna de data esteja no formato correto
df_vendas['Data da Venda'] = pd.to_datetime(df_vendas['Data da Venda'])

# 2. Filtrar as vendas do 2º trimestre de 2017 (Meses 4, 5 e 6)
vendas_2017 = df_vendas[
    (df_vendas['Data da Venda'].dt.year == 2017) & 
    (df_vendas['Data da Venda'].dt.month.isin([4, 5, 6]))
]['Qtd Vendida'].sum()

# 3. Filtrar as vendas do 2º trimestre de 2018 (Meses 4, 5 e 6)
vendas_2018 = df_vendas[
    (df_vendas['Data da Venda'].dt.year == 2018) & 
    (df_vendas['Data da Venda'].dt.month.isin([4, 5, 6]))
]['Qtd Vendida'].sum()

# 4. Calcular o crescimento percentual
crescimento = (vendas_2018 / vendas_2017) - 1

print(f"Total 2017: {vendas_2017}")
print(f"Total 2018: {vendas_2018}")
print(f"Crescimento: {crescimento:.1%}")

Total 2017: 4583
Total 2018: 6576
Crescimento: 43.5%


In [17]:
# 1. Unir as tabelas (mantendo o que já funcionou)
df_consolidado = df_vendas.merge(df_produto[['SKU', 'Preco Unitario', 'Custo Unitario']], on='SKU')

# 2. Calcular Faturamento e Custo por linha
# Importante: garantimos que são números
df_consolidado['Faturamento'] = df_consolidado['Qtd Vendida'] * df_consolidado['Preco Unitario']
df_consolidado['Custo Total'] = df_consolidado['Qtd Vendida'] * df_consolidado['Custo Unitario']

# 3. Agrupar por Loja somando APENAS as colunas que queremos (evita o erro de data/texto)
lojas_resumo = df_consolidado.groupby('Loja')[['Faturamento', 'Custo Total']].sum()

# 4. Calcular o Lucro e a Margem de Lucro por loja
lojas_resumo['Lucro'] = lojas_resumo['Faturamento'] - lojas_resumo['Custo Total']
lojas_resumo['Margem (%)'] = (lojas_resumo['Lucro'] / lojas_resumo['Faturamento']) * 100

# 5. Exibir o ranking da maior margem para a menor
ranking_margem = lojas_resumo.sort_values(by='Margem (%)', ascending=False)

print(ranking_margem)

                Faturamento  Custo Total       Lucro  Margem (%)
Loja                                                            
Curitiba         12272600.0    6628300.0   5644300.0   45.991070
Campinas         12332900.0    6663550.0   5669350.0   45.969318
Fortaleza        12210300.0    6609800.0   5600500.0   45.867014
Recife           12499650.0    6777200.0   5722450.0   45.780882
Salvador         11800950.0    6402800.0   5398150.0   45.743351
Guarulhos        23084900.0   12541200.0  10543700.0   45.673579
São Paulo        11268350.0    6125950.0   5142400.0   45.635785
Belo Horizonte   12272200.0    6672950.0   5599250.0   45.625479
Porto Alegre     12103950.0    6582600.0   5521350.0   45.616101
Rio de Janeiro   11701250.0    6369700.0   5331550.0   45.563935
Niterói          11554350.0    6309300.0   5245050.0   45.394592
Goiânia          12168500.0    6647800.0   5520700.0   45.368780
Nova Iguaço      12007950.0    6572400.0   5435550.0   45.266261


In [18]:
# 1. Trazer o Preco Unitario para a base de vendas
df_consolidado = df_vendas.merge(df_produto[['SKU', 'Preco Unitario']], on='SKU')

# 2. Calcular o faturamento de cada linha (Quantidade * Preço)
df_consolidado['Faturamento'] = df_consolidado['Qtd Vendida'] * df_consolidado['Preco Unitario']

# 3. Criar uma coluna apenas com o ANO da venda
df_consolidado['Ano'] = pd.to_datetime(df_consolidado['Data da Venda']).dt.year

# 4. Agrupar por Código Cliente e por Ano, somando o faturamento
gastos_cliente_ano = df_consolidado.groupby(['Código Cliente', 'Ano'])['Faturamento'].sum().reset_index()

# 5. Encontrar o valor máximo gasto por um cliente em um único ano
valor_maximo = gastos_cliente_ano['Faturamento'].max()

print(f"O valor máximo gasto por um cliente em 1 ano foi: R$ {valor_maximo:,.2f}")

O valor máximo gasto por um cliente em 1 ano foi: R$ 258,850.00
